In [2]:
import import_ipynb

In [1]:
import Dataset_Creation

In [6]:
import datetime # We need this for type hints

# --- Min-Heap Priority Queue Implementation ---
# (As built in our previous steps)

def heap_push(heap, item):
    """
    Pushes an item onto the heap (a list) while maintaining the
    min-heap property. This is the "sift-up" operation.
    """
    # 1. Add the new item to the end of the list
    heap.append(item)
    
    # 2. "Sift-up" the new item to its correct position
    pos = len(heap) - 1 # Start at the new item's position
    
    # While the item is not at the root (pos > 0) and
    # it's smaller than its parent
    while pos > 0:
        # Find the parent's position
        parent_pos = (pos - 1) // 2
        
        if heap[pos] < heap[parent_pos]:
            # Swap the item with its parent
            heap[pos], heap[parent_pos] = heap[parent_pos], heap[pos]
            # Move our position up to the parent's position
            pos = parent_pos
        else:
            # The item is in the correct place, so stop
            break

def heap_pop(heap):
    """
    Removes and returns the smallest item (the root) from the heap
    while maintaining the min-heap property. This is the "sift-down" operation.
    """
    if not heap:
        raise IndexError("pop from an empty heap")
        
    # 1. Store the root item (smallest) to return later
    return_item = heap[0]
    
    # 2. Get the last item from the heap
    last_item = heap.pop()
    
    # If the heap wasn't just the one item we popped,
    # we need to sift-down the 'last_item'
    if heap:
        # 3. Move the last item to the root
        heap[0] = last_item
        
        # 4. "Sift-down" the new root to its correct position
        pos = 0 # Start at the root
        last_index = len(heap) - 1
        
        while True:
            # Find children
            left_child_pos = 2 * pos + 1
            right_child_pos = 2 * pos + 2
            
            smallest_child_pos = pos # Assume parent is smallest
            
            # Check if left child exists and is smaller
            if left_child_pos <= last_index and heap[left_child_pos] < heap[smallest_child_pos]:
                smallest_child_pos = left_child_pos
                
            # Check if right child exists and is smaller
            if right_child_pos <= last_index and heap[right_child_pos] < heap[smallest_child_pos]:
                smallest_child_pos = right_child_pos
                
            # If the smallest is still the parent, we're done
            if smallest_child_pos == pos:
                break
            else:
                # Swap the parent with the smallest child
                heap[pos], heap[smallest_child_pos] = heap[smallest_child_pos], heap[pos]
                # Move our position down to the child's position
                pos = smallest_child_pos
    
    # 5. Return the original root
    return return_item

# --- End of Min-Heap Implementation ---

customer_df=Dataset_Creation.customer_df
# This assumes 'customer_df' exists from running the previous cell.

# 1. Create an empty list to use as our priority queue (min-heap)
customer_priority_queue = []

# Check if customer_df exists before using it
if 'customer_df' not in locals() and 'customer_df' not in globals():
    print("Error: 'customer_df' not found.")
    print("Please run the data generation cell (generate_dataset.py) first.")
else:
    print(f"Loading {len(customer_df)} customers into the priority queue...")

    # 2. Iterate through the DataFrame and add customers to the heap
    # We use itertuples() as it's a fast way to loop through a DataFrame
    for row in customer_df.itertuples():
        
        # 3. Create a tuple for each customer.
        # The heap will sort by the FIRST element, then the SECOND, and so on.
        # (Priority, Date of Admission, Customer ID, Name, ...)
        # This means Priority 1 is first.
        # If priorities are equal, the earlier admission date is first.
        
        # Column access with itertuples():
        # Index 0 is the DataFrame index (row.Index)
        #
        # --- Columns with spaces (become _N): ---
        # _1: "Customer ID"
        # _2: "Customer Name"
        # _5: "Date of Admission"
        # _6: "Days Admitted"
        # _7: "Current Status"
        #
        # --- Valid identifiers (accessed by name): ---
        # row.Priority
        # row.Department
        
        customer_tuple = (
            row.Priority,       # Sort by Priority (col 3)
            row._5, # Then by Date of Admission (col 5)
            row._1, # Then by Customer ID (col 1)
            row._2, # Customer Name (col 2)
            row.Department, # Department (col 4) <-- *** FIX: Was row._4 ***
            row._6, # Days Admitted (col 6)
            row._7  # Current Status (col 7)
        )
        
        # 4. Push the customer tuple onto the heap using OUR function
        heap_push(customer_priority_queue, customer_tuple)

    print("Priority queue created successfully.")
    print(f"Total customers in queue: {len(customer_priority_queue)}")


    # --- Let's test it! ---
    # 5. Let's "process" (pop) the 5 highest-priority customers
    print("\n--- Processing the Top 5 Highest-Priority Customers ---")

    # Ensure the queue is not empty before popping
    num_to_process = min(200, len(customer_priority_queue))

    if num_to_process == 0:
        print("Queue is empty, no customers to process.")
    else:
        for i in range(num_to_process):
            # Unpack the tuple
            # We use OUR function to pop from the heap
            (priority, date_adm, cust_id, name, department, days, status) = heap_pop(customer_priority_queue)
            
            print(f"  {i+1}. Processing: {cust_id} ({name})")
            print(f"     Priority: {priority}")
            print(f"     Admitted: {date_adm}")
            print(f"     Department: {department}")
            print(f"     Status:   {status}\n")

    print(f"Remaining customers in queue: {len(customer_priority_queue)}")



Loading 500 customers into the priority queue...
Priority queue created successfully.
Total customers in queue: 500

--- Processing the Top 5 Highest-Priority Customers ---
  1. Processing: C1105 (Joanna Morrison)
     Priority: 1
     Admitted: 2025-09-02
     Department: Orthopedics
     Status:   Admitted

  2. Processing: C1214 (Mark Arnold)
     Priority: 1
     Admitted: 2025-09-02
     Department: Dentist
     Status:   Admitted

  3. Processing: C1026 (Paula Vargas)
     Priority: 1
     Admitted: 2025-09-04
     Department: Cardiac
     Status:   Admitted

  4. Processing: C1155 (Anthony Baker)
     Priority: 1
     Admitted: 2025-09-04
     Department: Gynecology
     Status:   Admitted

  5. Processing: C1198 (Christopher Williams)
     Priority: 1
     Admitted: 2025-09-06
     Department: Oncology
     Status:   Admitted

  6. Processing: C1200 (Priscilla Greene)
     Priority: 1
     Admitted: 2025-09-06
     Department: Dentist
     Status:   Admitted

  7. Processing: C